In [0]:
from pyspark.sql.functions import col, substring, trim, current_timestamp, input_file_name

In [0]:
source_path = "/Volumes/oftalmo_sus/00_landing/raw_files/SIGTAP/tb_procedimento.txt"
target_path = "oftalmo_sus.01_bronze.bronze_dim_sigtap"

In [0]:
df_raw = (
    spark.read
    .format("csv")
    .option("encoding", "windows-1252") # O encoding oficial do Windows/DATASUS
    .option("delimiter", "\x01")        # Um caractere invisível que não existe no arquivo, forçando a ler a linha inteira
    .load(source_path)
    .withColumnRenamed("_c0", "value")  # Renomeia para 'value' para manter a lógica
)

In [0]:
#Parsing Posicional
df_bronze = (
    df_raw
    .select(
        # substring(coluna, posicao_inicial, tamanho)
        substring(col("value"), 1, 10).alias("CO_PROCEDIMENTO"),
        substring(col("value"), 11, 250).alias("NO_PROCEDIMENTO"),
        substring(col("value"), 261, 1).alias("TP_COMPLEXIDADE")
    )
    # removendo os espaços sobram no final dos textos
    .withColumn("CO_PROCEDIMENTO", trim(col("CO_PROCEDIMENTO")))
    .withColumn("NO_PROCEDIMENTO", trim(col("NO_PROCEDIMENTO")))
    
    # adicionando metadados de governança
    .withColumn("data_ingestao_bronze", current_timestamp())
    .withColumn("arquivo_origem", col("_metadata.file_path"))
)

display(df_bronze.limit(15))

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite") 
    .saveAsTable(target_path)
)